# Importação de Pacotes

In [ ]:
#leitura da base de dados
import pandas as pd
from pathlib import Path
import parquet

#modelos preditivos escolhidos
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

#validação cruzada
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, HalvingGridSearchCV
import numpy as np

#métricas
import matplotlib
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, classification_report

#pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
def estimadores(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)

    acuracia = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    roc_auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class='ovr',
        average='weighted'
    )

    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test)

    print(f"""
        Acurácia: {acuracia:.3f}
        Recall (weighted): {recall:.3f}
        F1-score (weighted): {f1:.3f}
        ROC AUC (ovr): {roc_auc:.3f}
        """)

    print(classification_report(y_test, y_pred))

## Leitura DataFrame

In [ ]:
direcao = Path("../..") / "data"
caminho = direcao / "erro_medico_tidy_final.parquet"


df_erro_simples = pd.read_parquet(caminho)

df = df_erro_simples

In [ ]:
df.head(2)

In [ ]:
df.columns

# Aplicação de Pipelines

### CBC

In [ ]:
modelo= CatBoostClassifier(auto_class_weights='Balanced')

parametros = {
    'modelo__iterations': [300],
    'modelo__depth': [4, 6, 8],
    'modelo__learning_rate': [0.05, 0.1],
    'modelo__verbose': [0]
}

In [ ]:
stopwords = ['a', 'à', 'o', 'e', 'de', 'do', 'da', 'em', 'um', 'uma', 'para', 'por', 'com', 
            'não', 'que', 'é', 'são', 'se', 'na', 'no', 'ao', 'como', 'mais', 'mas', 'ou',
            'ao', 'até', 'pela', 'pelas', 'pelo', 'pelos', 'nos', 'nas', 'deles', 'dela',
            'ele', 'ela', 'eles', 'elas', 'você', 'vocês', 'nós', 'eu', 'isso', 'isto',
            'aos', 'das', 'dos', 'esse', 'essa', 'esses', 'essas', 'aquele', 'aquela']


stopwords_jur = ['autor', 'réu', 'reu', 'partes', 'processo', 'autos', 'ação', 'acao',
                'decisão', 'decisao', 'sentença', 'sentenca', 'acórdão', 'acordao',
                'juiz', 'juízo', 'juizo', 'tribunal', 'vara', 'comarca','relator', 'relatora', 
                'ministro', 'ministra', 'requerente', 'requerido', 'requerida', 'apelante', 'apelado',
                'apelada', 'impetrante', 'impetrado', 'agravante', 'agravado', 'petição', 'peticao', 
                'inicial', 'contestação', 'contestacao', 'recurso', 'fundamentação', 'fundamentacao',
                'dispositivo', 'ementa', 'direito', 'fato', 'fatos', 'termos', 'forma', 'caso', 'art', 'artigo', 
                'lei', 'código', 'codigo', 'inciso', 'parágrafo', 'paragrafo', 'caput', 'diante', 'face', 'ante',
                'presente', 'autos', 'mesmos','bem', 'assim', 'ainda', 'conforme', 'respectivamente','visto', 
                'vista', 'decide', 'julgo', 'julgar', 'julga']

conjunto_stop_words = stopwords + stopwords

lista_X = ['valor_acao','n_autores', 'n_reus', 'tem_hospital', 'tem_plano_saude',
           'tem_ente_publico', 'tem_medico_individual', 'n_adv_autor', 'n_adv_reu',
           'tem_perito', 'tem_denuncia_lide', 'tem_assistente', 'resumo_caso']

X = df[lista_X]

y = df["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)

tokenizer = Pipeline([
    ('token', TfidfVectorizer(
        max_features=10000, 
        ngram_range=(1, 2),
        stop_words= conjunto_stop_words,
        sublinear_tf=True))])

preprocessor = ColumnTransformer(
    transformers=[
        ('tokenizer', tokenizer, "resumo_caso")],
        remainder='passthrough')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='f1_macro',
    refit= True,
    cv=5,
    verbose=2
)

searchCV_pipeline.fit(X_train, y_train)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)

### GBC

In [ ]:
modelo= GradientBoostingClassifier()

parametros = {
    'modelo__n_estimators': [100],
    'modelo__max_depth': [2, 6],
    'modelo__max_leaf_nodes': [10],
    'modelo__learning_rate': [0.05, 0.2]
    }

In [ ]:
stopwords = ['a', 'à', 'o', 'e', 'de', 'do', 'da', 'em', 'um', 'uma', 'para', 'por', 'com', 
            'não', 'que', 'é', 'são', 'se', 'na', 'no', 'ao', 'como', 'mais', 'mas', 'ou',
            'ao', 'até', 'pela', 'pelas', 'pelo', 'pelos', 'nos', 'nas', 'deles', 'dela',
            'ele', 'ela', 'eles', 'elas', 'você', 'vocês', 'nós', 'eu', 'isso', 'isto',
            'aos', 'das', 'dos', 'esse', 'essa', 'esses', 'essas', 'aquele', 'aquela']
#adicionar stopword jurídicas


lista_X = ['valor_acao','n_autores', 'n_reus', 'tem_hospital', 'tem_plano_saude',
           'tem_ente_publico', 'tem_medico_individual', 'n_adv_autor', 'n_adv_reu',
           'tem_perito', 'tem_denuncia_lide', 'tem_assistente', 'resumo_caso']

X = df[lista_X]

y = df["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)


tokenizer = Pipeline([
    ('token', TfidfVectorizer(
        max_features=10000, 
        ngram_range=(1, 2),
        stop_words= stopwords,
        sublinear_tf=True))])


num_prep = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_prep, ["valor_acao"]),
        ('tokenizer', tokenizer, "resumo_caso")], 
        remainder='passthrough')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='roc_auc_ovr',
    refit= True,
    cv=5,
    verbose=2
)

# balanceamento por sample_weight (GradientBoostingClassifier não aceita class_weight)
sample_weight = compute_sample_weight('balanced', y_train)
searchCV_pipeline.fit(X_train, y_train, modelo__sample_weight=sample_weight)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)